# Retrieval-Augmented Generation (RAG)

In [1]:
from setup import bedrock_tool
import chromadb
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

✅ AWS credentials found
✅ OpenAI credentials found
✅ EXA credentials found


Create a static calorie table that we can use as a tool:

In [2]:
# We populated the RAG with the data from the data/calories.csv file in
# the ../rag_setup/rag_setup.ipynb notebook

chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [11]:
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")

In [3]:
results = nutrition_db.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):   
    print(doc)
    print("\n")

2026-03-02 13:48:45.229852762 [W:onnxruntime:Default, device_discovery.cc:131 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename ""5620e0c7-8062-4dce-aeb7-520c7ef76171"" dit not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Food: Banana
        Category: Fruits
        Nutritional Information:
        - Calories: 89 per 100g
        - Energy: 374 kJ per 100g
        - Serving size reference: 100g

        This is a fruits food item that provides 89 calories per 100 grams.


Food: Banana Juice
        Category: (Fruit)Juices
        Nutritional Information:
        - Calories: 50 per 100g
        - Energy: 210 kJ per 100g
        - Serving size reference: 100ml

        This is a (fruit)juices food item that provides 50 calories per 100 grams.




In [12]:
results = nutrition_qna.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):   
    print(doc)
    print("\n")

Question: What is the recommended amount of bananas I should consume to count it as a single serving?
        Answer: One small-sized banana can be counted as a single serving.

        This Q&A pair provides information about nutrition and health topics.


Question: Which food is recommended for infants after they've been introduced to ripe bananas and sweet potatoes?
        Answer: Introduce porridge made from wheat flour or ground rice, starting with only one cereal. Once a week has passed, you may increase the frequency of this new food to two feedings per day.

        This Q&A pair provides information about nutrition and health topics.




In [ ]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

In [13]:
@function_tool
def qna_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up nutritional question and answer information.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_qna.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition qna information found for: {query}"


    return "Nutrition Information:\n" + "\n".join(results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [ ]:
# print(calorie_lookup_tool('bananas'))

Nutrition Information:
Banana (Fruits): 89.0 calories per 100g
Banana Juice ((Fruit)Juices): 50.0 calories per 100g
Banana Nut Bread (Pastries,Breads&Rolls): 326.0 calories per 100g


In [ ]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(calorie_lookup_tool.__dict__), bedrock_tool(qna_lookup_tool.__dict__)],
)

In [ ]:
with trace("Nutrition Assistant with RAG") as t:
    result = await Runner.run(
        calorie_agent,
        "Give me nutritional information about bananas and apples. Also give calories per 100g",
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: Based on the tool results, here's the nutritional information for bananas:

- Banana (Fruits): 89.0 calories per 100g
- Banana Juice ((Fruit)Juices): 50.0 calories per 100g
- Banana Nut Bread (Pastries,Breads&Rolls): 326.0 calories per 100g

Please note that these are approximate values and can vary based on ripeness and variety.

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_79abc902b96440fbb9e59d873f6817b6
